# Phutball GRPO pilot: Qwen3.5-9B (thinking on) + LoRA, rewarded only by the engine

The thinking is free-form: the reward checks the final answer with the phutball engine (1 if verified), gives a small
bonus to correct samples shorter than their siblings, and -0.5 to samples cut off at the token cap. No format, language or
readability term, so the thinking may drift ("neuralese"); the drift is logged.

**Runtime:** A100 GPU, High-RAM. **Cost:** smoke test ~0.3 h, pilot 2.5 h (stops itself), eval ~0.7 h: about 3.5 h, ~24 units.
Everything is saved to Drive (`MyDrive/phutball/grpo_pilot/`), and the pilot resumes from its last checkpoint if the
runtime drops: just run the cells again. **Disconnect the runtime when done.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/phutball/grpo_pilot'; os.makedirs(OUT, exist_ok=True)
BENCH_OUT = '/content/drive/MyDrive/phutball/llm_bench'          # the zero-shot results (reference)
RUN = f'{OUT}/run1'; CAP = 6144
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention   # fast Gated DeltaNet kernels for Qwen3.5 (else a slow PyTorch fallback)
!pip -q uninstall -y torchao   # Colab ships torchao 0.10; peft refuses versions < 0.16 (and we do not use it)
!python -c "import trl, vllm, peft, transformers, torch; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__, 'torch', torch.__version__)"

## 1. Smoke test (2 tiny steps, ~15 min incl. the model download)
Catches setup errors before the long run. If it fails, paste the last 40 lines to Claude.

In [ ]:
!python -m llm_train.grpo_pilot --out /content/smoke --smoke 2>&1 | tail -40

## 2. Pilot (2.5 h of training, then it saves and stops)
4 puzzles x 8 samples per step, thinking capped at 6,144 tokens. Prints one reward line per generation batch and the
minutes per step. Full log: `RUN/train.log`; every sample: `RUN/rollouts.jsonl`.

In [ ]:
os.makedirs(RUN, exist_ok=True)
!python -m llm_train.grpo_pilot --out {RUN} --hours 2.5 --cap {CAP} 2>&1 | tee -a {RUN}/train.log | grep --line-buffered -E "reward call|\[step|rror|Traceback|saved|skipped|puzzles|trl|forcing|OutOfMemory"

## 3. Training curves and the drift of the thinking

In [ ]:
!python -m llm_train.analyze_rollouts {RUN}/rollouts.jsonl --plot {RUN}/curves.png --window 3
from IPython.display import Image, display
display(Image(f'{RUN}/curves.png'))

## 4. Eval: pilot vs the untrained model at the same thinking budget (275 benchmark items)
The reference is the zero-shot thinking-on run (24k tokens) cut at the same budget and budget-forced, so both are
"thinking on, 6,144 tokens".

In [ ]:
import subprocess, time, urllib.request
def serve(model, max_len=CAP + 2048):
    !pkill -f "vllm serve" ; sleep 5
    log = open('/content/vllm_serve.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', model, '--max-model-len', str(max_len), '--gpu-memory-utilization', '0.90',
                          '--port', '8000'], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up:', model); return p
        except Exception: time.sleep(5)
    raise RuntimeError('vLLM did not start: see /content/vllm_serve.log')

!python -m llm_train.merge_lora --adapter {RUN}/final --out /content/merged 2>&1 | tail -3
server = serve('/content/merged')
!python -m llm_bench.run --model /content/merged --name pilot-run1 --thinking true --max-tokens {CAP} --per-group 25 --concurrency 64
!python -m llm_bench.run --model /content/merged --thinking true --salvage pilot-run1 --per-group 25 --concurrency 32
server = serve('Qwen/Qwen3.5-9B')
!cp {BENCH_OUT}/qwen3.5-9b-think.jsonl llm_bench/results/
!python -m llm_bench.run --model Qwen/Qwen3.5-9B --thinking true --salvage qwen3.5-9b-think --truncate-at {CAP} --per-group 25 --concurrency 32
!pkill -f "vllm serve"
!cp llm_bench/results/pilot-run1* llm_bench/results/qwen3.5-9b-think-at{CAP}* {RUN}/

In [ ]:
import json
A = json.load(open(f'{RUN}/qwen3.5-9b-think-at{CAP}-forced.summary.json')); B = json.load(open(f'{RUN}/pilot-run1-forced.summary.json'))
print(f"{'':28s} {'untrained':>10s} {'pilot':>10s}   (thinking on, {CAP} tokens, budget-forced; truncated share in brackets)")
for k in ('win (all)', 'win: near-miss negative', 'win: positive, 1-jump', 'win: positive, 2-jump', 'win: positive, 3-jump',
          'win: positive, 4-jump', 'forced placement', 'block placement', 'prevent placement'):
    if k in A and k in B:
        print(f"{k:28s} {A[k]['accuracy']:6.1%} ({A[k].get('truncated', 0):4.0%}) {B[k]['accuracy']:6.1%} ({B[k].get('truncated', 0):4.0%})")
print('mean completion tokens', A.get('mean_completion_tokens'), B.get('mean_completion_tokens'))